<a href="https://colab.research.google.com/github/CMP7005/AQI_India_repo/blob/master/Week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 81.2 MB/s eta 0:00:00


In [2]:
import os
os.makedirs("air_quality_app/pages", exist_ok=True)

In [3]:
%%writefile air_quality_app/app.py
import streamlit as st

st.set_page_config(page_title="Air Quality App")

st.title("Air Quality Analytics App")

st.write("""
This application contains three sections:

1️⃣ Upload Dataset
2️⃣ Data Description and Preprocessing
3️⃣ Exploratory Data Analysis (EDA)
4️⃣ Model Building
""")

st.info("Use the sidebar to navigate between pages.")

Writing air_quality_app/app.py


In [4]:
%%writefile air_quality_app/pages/0_Data_upload.py
# Import required libraries
import streamlit as st   # Streamlit for building the web app
import pandas as pd      # Pandas for data handling
import os                # OS module to check if files exist

# Display the title on the Streamlit page
st.title("Upload Dataset")

# Define the file name where the uploaded dataset will be saved
DATA_PATH = "uploaded_data.csv"

# Create a file uploader widget that accepts CSV files
file = st.file_uploader("Upload CSV dataset", type=["csv"])

# If the user uploads a file
if file is not None:

    # Read the uploaded CSV file into a pandas dataframe
    df = pd.read_csv(file)

    # Save the uploaded dataset to the local folder
    # This allows the app to reuse the dataset even after refresh
    df.to_csv(DATA_PATH, index=False)

    # Show a success message on the app
    st.success("Dataset uploaded successfully!")

    # Display a small section header
    st.subheader("Preview")

    # Show the first 5 rows of the dataset in an interactive table
    st.dataframe(df.head())

# If no file is uploaded but a dataset already exists locally
elif os.path.exists(DATA_PATH):

    # Load the previously saved dataset
    df = pd.read_csv(DATA_PATH)

    # Inform the user that the dataset has already been uploaded earlier
    st.info("Dataset already uploaded")

    # Show preview header
    st.subheader("Preview")

    # Display the first 5 rows of the dataset
    st.dataframe(df.head())

# If no dataset exists and nothing has been uploaded
else:

    # Show a warning asking the user to upload a dataset
    st.warning("Please upload a dataset to begin.")

Writing air_quality_app/pages/0_Data_upload.py


In [5]:
%%writefile air_quality_app/pages/1_Data_Description.py

# Import required libraries
import streamlit as st   # Streamlit is used to create the web application interface
import pandas as pd      # Pandas is used for data manipulation and analysis
import os                # OS module helps check if files exist in the system


# Display the page title in the Streamlit app
st.title("Data Description")


# Check if the uploaded dataset exists
# If the dataset exists, load it using pandas
if os.path.exists("uploaded_data.csv"):
    df = pd.read_csv("uploaded_data.csv")

# If the dataset does not exist, show a warning message
# and stop the execution of the app
else:
    st.warning("Please upload data first on the Upload Data page.")
    st.stop()


# ---------------------------------------------------------
# Dataset Preview
# ---------------------------------------------------------

# Display a subheader for the dataset preview section
st.subheader("Dataset Preview")

# Show the first 5 rows of the dataset in an interactive table
st.dataframe(df.head())


# ---------------------------------------------------------
# Dataset Shape
# ---------------------------------------------------------

# Display the number of rows and columns in the dataset
st.subheader("Dataset Shape")

# df.shape[0] gives the number of rows
st.write("Rows:", df.shape[0])

# df.shape[1] gives the number of columns
st.write("Columns:", df.shape[1])


# ---------------------------------------------------------
# Column Names
# ---------------------------------------------------------

# Display all column names in the dataset
st.subheader("Column Names")

# Convert column names to a list and display them
st.write(list(df.columns))


# ---------------------------------------------------------
# Missing Values
# ---------------------------------------------------------

# Display the number of missing values in each column
st.subheader("Missing Values")

# df.isnull().sum() counts missing values in each column
# reset_index() converts the result into a dataframe
# rename() changes column names for better readability
st.dataframe(
    df.isnull().sum().reset_index().rename(
        columns={"index": "Column", 0: "Missing Values"}
    )
)


# ---------------------------------------------------------
# Data Preprocessing
# ---------------------------------------------------------

# Create a copy of the dataset and remove rows with missing values
processed_df = df.copy().dropna()

# Save the cleaned dataset for use in other pages of the Streamlit app
processed_df.to_csv("processed_data.csv", index=False)


# ---------------------------------------------------------
# Display the processed dataset
# ---------------------------------------------------------

# Show preview of the cleaned dataset
st.subheader("Preprocessed Dataset Preview")
st.dataframe(processed_df.head())

# Show the shape of the cleaned dataset
st.subheader("Preprocessed Dataset Shape")
st.write("Rows:", processed_df.shape[0])
st.write("Columns:", processed_df.shape[1])


Writing air_quality_app/pages/1_Data_Description.py


In [6]:
%%writefile air_quality_app/pages/2_EDA.py
# Import required libraries
import streamlit as st   # Streamlit is used to build the interactive web app
import pandas as pd      # Pandas is used for data manipulation
import os                # OS module helps check if files exist


# Display the page title
st.title("Exploratory Data Analysis")


# ---------------------------------------------------------
# Check if the processed dataset exists
# ---------------------------------------------------------

# If the cleaned dataset is available, load it
if os.path.exists("processed_data.csv"):

    # Read the preprocessed dataset
    df = pd.read_csv("processed_data.csv")

    # Display a preview of the dataset
    st.subheader("Preprocessed Data Preview")
    st.dataframe(df.head())


    # ---------------------------------------------------------
    # Select numeric columns for analysis
    # ---------------------------------------------------------

    # Identify numeric columns in the dataset
    # Only numerical columns are suitable for charts and correlation analysis
    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()


    # Check if numeric columns exist
    if numeric_cols:

        # ---------------------------------------------------------
        # Single column visualisations
        # ---------------------------------------------------------

        # Create a dropdown menu for selecting a numeric column
        column = st.selectbox("Select a column", numeric_cols)

        # Display a bar chart for the selected column
        st.subheader("Bar Chart")
        st.bar_chart(df[column])

        # Display a line chart for the selected column
        st.subheader("Line Chart")
        st.line_chart(df[column])


        # ---------------------------------------------------------
        # Scatter plot (relationship between two variables)
        # ---------------------------------------------------------

        # Create dropdown menus for selecting X and Y axes
        x_axis = st.selectbox("Select X-axis", numeric_cols, index=0)
        y_axis = st.selectbox("Select Y-axis", numeric_cols, index=min(1, len(numeric_cols)-1))

        # Ensure the same column is not selected for both axes
        if x_axis != y_axis:

            # Display scatter plot showing relationship between two variables
            st.subheader("Scatter Plot")
            st.scatter_chart(df[[x_axis, y_axis]])


        # ---------------------------------------------------------
        # Correlation Matrix
        # ---------------------------------------------------------

        # Display correlation matrix for all numeric columns
        # Correlation shows relationships between variables (-1 to +1)
        st.subheader("Correlation Matrix")
        st.dataframe(df[numeric_cols].corr())


    # If no numeric columns are found
    else:
        st.error("No numeric columns found in the dataset.")


# If the processed dataset does not exist
else:

    # Ask the user to upload and preprocess the dataset first
    st.warning("Please upload and preprocess the dataset first on the Data Description page.")

Writing air_quality_app/pages/2_EDA.py


In [7]:
%%writefile air_quality_app/pages/3_Model_Building.py
# Import required libraries
import streamlit as st                     # Streamlit for creating the web application
import pandas as pd                        # Pandas for handling data
import os                                  # OS module to check if files exist
from sklearn.model_selection import train_test_split  # Used to split dataset into training and testing sets
from sklearn.linear_model import LinearRegression     # Linear Regression model


# Display the page title
st.title("Model Building")


# ---------------------------------------------------------
# Check if the processed dataset exists
# ---------------------------------------------------------

# If the cleaned dataset exists, load it
if os.path.exists("processed_data.csv"):

    # Read the processed dataset
    df = pd.read_csv("processed_data.csv")

    # Display a preview of the dataset
    st.subheader("Preprocessed Data")
    st.dataframe(df.head())


    # ---------------------------------------------------------
    # Select numeric columns for model building
    # ---------------------------------------------------------

    # Identify numeric columns (only numerical variables can be used in regression)
    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()


    # Ensure there are enough numeric columns for building a model
    if len(numeric_cols) >= 2:

        # Allow the user to select the target variable (the variable to predict)
        target = st.selectbox("Select target variable", numeric_cols)


        # Allow the user to select feature variables (input variables used for prediction)
        features = st.multiselect(
            "Select feature variables",
            [col for col in numeric_cols if col != target]   # Exclude the target column from features
        )


        # Proceed only if at least one feature is selected
        if features:

            # Define input features (X) and target variable (y)
            X = df[features]
            y = df[target]


            # ---------------------------------------------------------
            # Split the dataset into training and testing sets
            # ---------------------------------------------------------

            # 80% of the data is used for training and 20% for testing
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )


            # ---------------------------------------------------------
            # Train the Linear Regression model
            # ---------------------------------------------------------

            # Create the regression model
            model = LinearRegression()

            # Train the model using the training data
            model.fit(X_train, y_train)


            # ---------------------------------------------------------
            # Evaluate model performance
            # ---------------------------------------------------------

            # Calculate the R² score using the test dataset
            score = model.score(X_test, y_test)

            # Display model performance
            st.subheader("Model Performance")
            st.write("R² Score:", score)


        # If no features are selected
        else:
            st.info("Please select at least one feature.")


    # If there are not enough numeric columns
    else:
        st.error("Not enough numeric columns for model building.")


# If the processed dataset does not exist
else:

    # Ask the user to upload and preprocess the dataset first
    st.warning("Please upload and preprocess the dataset first on the Data Description page.")

Writing air_quality_app/pages/3_Model_Building.py


In [8]:
!ls

air_quality_app  drive	sample_data


In [ ]:
!ls air_quality_app

app.py	pages


In [9]:
!ls /content/air_quality_app/pages

0_Data_upload.py  1_Data_Description.py  2_EDA.py  3_Model_Building.py


In [10]:
!wget -q -O - ipv4.icanhazip.com

34.26.173.53


In [ ]:
!streamlit run air_quality_app/app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴your url is: https://violet-cities-sell.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.26.173.53:8501

